# Conception du réseau de neurones LeNet-5

#### Définition des fonctions nécessaires


In [ ]:
import numpy as np

### La couche C1

In [5]:


class Conv2D:
    """
    Couche de convolution.

    Paramètres
    ----------
    in_channels  : canaux de l'image en entrée
    out_channels : nombre de filtres
    kernel_size  : côté du filtre carré

    Attributs appris
    ----------------
    W : poids   
    b : biais   
    """

    # ──────────────────────────────────────────────────────────────
    # INITIALISATION
    # ──────────────────────────────────────────────────────────────
    def __init__(self, in_channels, out_channels, kernel_size):
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size  = kernel_size

    
        fan_in  = in_channels  * kernel_size * kernel_size   # connexions entrantes
        fan_out = out_channels * kernel_size * kernel_size   # connexions sortantes
        std = np.sqrt(2.0 / (fan_in + fan_out))

        # Poids : un filtre (in_ch × k × k) par out_channel
        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * std
        # Biais : un scalaire par filtre
        self.b = np.zeros(out_channels)

        # Gradients, remplis pendant backward
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    # ──────────────────────────────────────────────────────────────
    # FORWARD  — fait glisser chaque filtre sur l'image
    #
    #  entrée  x   : (batch, in_ch, H,     W    )
    #  sortie out  : (batch, out_ch, H_out, W_out)
    #  avec H_out  = H - k + 1  
    # ──────────────────────────────────────────────────────────────
    def forward(self, x):

        self.x = x   # sauvegarde nécessaire pour le backward

        batch, in_c, H, W = x.shape
        k     = self.kernel_size
        H_out = H - k + 1
        W_out = W - k + 1

        out = np.zeros((batch, self.out_channels, H_out, W_out))

        # Pour chaque filtre f, on parcourt toutes les positions (i, j)
        for f in range(self.out_channels):
            for i in range(H_out):
                for j in range(W_out):

                    # Fenêtre courante de l'image — shape (batch, in_c, k, k)
                    region = x[:, :, i:i+k, j:j+k]

                    # Produit élément par élément → somme → + biais
                    # résultat : un scalaire par image du batch
                    out[:, f, i, j] = np.sum(region * self.W[f], axis=(1,2,3)) + self.b[f]

        return out   # shape (batch, out_ch, H_out, W_out)

    # ──────────────────────────────────────────────────────────────
    # BACKWARD — remonte le gradient et calcule les corrections
    # ──────────────────────────────────────────────────────────────
    def backward(self, d_out):

        x = self.x
        batch, in_c, H, W = x.shape
        k     = self.kernel_size
        H_out = d_out.shape[2]
        W_out = d_out.shape[3]

        # Remise à zéro des gradients avant accumulation
        dx       = np.zeros_like(x)
        self.dW  = np.zeros_like(self.W)
        self.db  = np.zeros_like(self.b)

        for f in range(self.out_channels):
            for i in range(H_out):
                for j in range(W_out):

                    region = x[:, :, i:i+k, j:j+k]  # même fenêtre que forward

                    # dL/dW — le filtre doit changer d'autant que
                    # l'entrée était forte ET l'erreur grande
                    # += car le même filtre a glissé sur toute l'image
                    self.dW[f] += np.sum(
                        region * d_out[:, f, i, j][:, None, None, None],
                        axis=0
                    )

                    # dL/db — biais ajouté partout → correction = somme des erreurs
                    self.db[f] += np.sum(d_out[:, f, i, j])

                    # dL/dx — chaque pixel a pu être couvert par le filtre
                    # à plusieurs positions → on accumule toutes ses responsabilités
                    dx[:, :, i:i+k, j:j+k] += (
                        self.W[f] * d_out[:, f, i, j][:, None, None, None]
                    )

        return dx   # transmis à la couche précédente (AvgPool ou entrée)

### AvgPool2D

In [ ]:
class AvgPool2D:
    """
    Average Pooling 2x2 avec stride=2.
    Divise les dimensions H et W par 2.
    """
    def __init__(self, pool_size=2):
        self.pool_size = pool_size

    def forward(self, x):
        """
        x shape   : (batch, channels, H, W)
        out shape : (batch, channels, H//2, W//2)
        """
        self.x = x
        batch, c, H, W = x.shape
        p = self.pool_size
        H_out = H // p
        W_out = W // p

        out = np.zeros((batch, c, H_out, W_out))
        for i in range(H_out):
            for j in range(W_out):
                # Moyenne sur la fenêtre 2x2
                out[:, :, i, j] = np.mean(
                    x[:, :, i*p:(i+1)*p, j*p:(j+1)*p],
                    axis=(2, 3)
                )
        return out

    def backward(self, d_out):
        """
        Le gradient se redistribue équitablement sur les 4 cases de la fenêtre.
        """
        x = self.x
        batch, c, H, W = x.shape
        p = self.pool_size
        H_out = d_out.shape[2]
        W_out = d_out.shape[3]

        dx = np.zeros_like(x)
        for i in range(H_out):
            for j in range(W_out):
                # Chaque case reçoit 1/4 du gradient
                dx[:, :, i*p:(i+1)*p, j*p:(j+1)*p] = (
                    d_out[:, :, i, j][:, :, None, None] / (p * p)
                )
        return dx

### Activation

In [7]:
class Tanh:
    """
    Activation Tanh : f(x) = (e^x - e^-x) / (e^x + e^-x)
    Dérivée         : f'(x) = 1 - tanh(x)²
    Utilisée dans le LeNet original (LeCun 1998)
    """
    def forward(self, x):
        self.out = np.tanh(x)
        return self.out

    def backward(self, d_out):
        return d_out * (1 - self.out ** 2)


class ReLU:
    """
    Activation ReLU : f(x) = max(0, x)
    Dérivée         : 1 si x > 0, sinon 0
    Utilisée pour la version moderne de LeNet
    """
    def forward(self, x):
        self.x = x
        return np.maximum(0, x)

    def backward(self, d_out):
        return d_out * (self.x > 0)

### Flatten + Linear : pour faire le lien entre les convolutions et les couches denses

In [8]:
class Flatten:
    """
    Transforme (batch, channels, H, W) en (batch, channels*H*W)
    """
    def forward(self, x):
        self.shape = x.shape
        return x.reshape(x.shape[0], -1)

    def backward(self, d_out):
        return d_out.reshape(self.shape)


class Linear:
    """
    Couche linéaire (dense, fully connected).
     Y = X·W + b
    Paramètres:
        in_features  : taille du vecteur d'entrée
        out_features : taille du vecteur de sortie
    """
    def __init__(self, in_features, out_features):
        # Initialisation Xavier
        std = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features) * std
        self.b = np.zeros(out_features)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b   # @ = produit matriciel

    def backward(self, d_out):
        self.dW = self.x.T @ d_out
        self.db = np.sum(d_out, axis=0)
        return d_out @ self.W.T

## Le modèle LeNet-5

In [9]:
class LeNet5:
    """
    Architecture LeNet-5 (LeCun, 1998).
    
    Input  : (batch, 1, 32, 32)
    Output : (batch, 10) :  scores pour chaque chiffre 0-9
    
    Architecture :
        C1  : Conv(1,6,  5x5) + Tanh
        S2  : AvgPool(2x2)
        C3  : Conv(6,16, 5x5) + Tanh
        S4  : AvgPool(2x2)
        C5  : Conv(16,120, 5x5) + Tanh
        F6  : Linear(120,84) + Tanh
        Out : Linear(84,10)
    """
    def __init__(self):
        # --- Couches convolutives ---
        self.C1  = Conv2D(in_channels=1,   out_channels=6,   kernel_size=5)
        self.A1  = Tanh()

        self.S2  = AvgPool2D(pool_size=2)

        self.C3  = Conv2D(in_channels=6,   out_channels=16,  kernel_size=5)
        self.A3  = Tanh()

        self.S4  = AvgPool2D(pool_size=2)

        self.C5  = Conv2D(in_channels=16,  out_channels=120, kernel_size=5)
        self.A5  = Tanh()

        # --- Flatten ---
        self.F   = Flatten()

        # --- Couches fully connected ---
        self.F6  = Linear(in_features=120, out_features=84)
        self.A6  = Tanh()

        self.Out = Linear(in_features=84,  out_features=10)

        # Liste de toutes les couches (utile pour l'optimizer)
        self.layers = [
            self.C1, self.A1,
            self.S2,
            self.C3, self.A3,
            self.S4,
            self.C5, self.A5,
            self.F,
            self.F6, self.A6,
            self.Out
        ]

    def forward(self, x):
        """
        Passe avant : x traverse toutes les couches dans l'ordre.
        """
        out = x
        for layer in self.layers:
            out = layer.forward(out)
        return out   # shape : (batch, 10) — scores bruts (logits)

    def backward(self, d_out):
        """
        Passe arrière : le gradient remonte dans l'ordre inverse.
        """
        grad = d_out
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    def get_params(self):
        """
        Retourne toutes les couches qui ont des poids W et b.
        Utilisé par l'optimizer pour mettre à jour les poids.
        """
        return [l for l in self.layers if hasattr(l, 'W')]

 ### La Loss (Softmax + Cross-Entropy)

In [ ]:
def softmax(x):
    """
    Softmax numériquement stable.
    On soustrait le max pour éviter les overflow.
    
    """
    x_shifted = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


def cross_entropy_loss(logits, y_true):
    """
    Calcule la Cross-Entropy Loss + son gradient.
    
    Paramètres:
        logits : scores bruts du modèle, shape (batch, 10)
        y_true : labels vrais, shape (batch,)  ex: [3, 7, 1, ...]
    
    Retourne:
        loss  : scalaire (moyenne sur le batch)
        d_out : gradient de la loss par rapport aux logits
    
    Astuce mathématique : la combinaison Softmax + Cross-Entropy
    donne un gradient très simple = (proba_prédite - one_hot_vrai)
    """
    batch = logits.shape[0]
    probs = softmax(logits)          # (batch, 10) → probabilités

    # On récupère la proba de la vraie classe pour chaque exemple
    correct_probs = probs[np.arange(batch), y_true]   # (batch,)

    # Loss = -log(proba de la bonne classe), moyennée sur le batch
    loss = -np.mean(np.log(correct_probs + 1e-9))     # 1e-9 évite log(0)

    # Gradient = proba - 1 pour la vraie classe, proba sinon
    d_out = probs.copy()
    d_out[np.arange(batch), y_true] -= 1
    d_out /= batch                   # on divise par batch (cohérent avec mean)

    return loss, d_out

### L'Optimizer (SGD)

In [ ]:
class SGD:
    """
    Stochastic Gradient Descent avec learning rate fixe.
    
    Règle de mise à jour : W = W - lr * dW
    """
    def __init__(self, params, lr=0.01):
        self.params = params   # liste des couches avec poids
        self.lr = lr

    def step(self):
        for layer in self.params:
            layer.W -= self.lr * layer.dW
            layer.b -= self.lr * layer.db

### Fonction d'accuracy

In [ ]:
def accuracy(logits, y_true):
    """
    Calcule le pourcentage de bonnes prédictions.
    La prédiction = classe avec le score le plus élevé.
    """
    preds = np.argmax(logits, axis=1)   # indice du max sur les 10 classes
    return np.mean(preds == y_true)

### Test du modèle complet